# Assignment 3: Normalizing Flows for Uncertainty Prediction

In this notebook, I use the astronomy spectra dataset to predict the three stellar labels temperature, gravity, and metallicity together with their uncertainties using normalizing flows.

I implement and compare three conditional density models with the Jammy Flows library:

1. a diagonal Gaussian flow  
2. a full 3D Gaussian flow  
3. a Gaussianization flow with an affine flow for more flexible, non-Gaussian PDFs  

The models are trained with the negative log-likelihood loss.  
Their uncertainty quality is evaluated using pull distributions, and a few predicted PDFs are visualized for the full-flow model.

In [ ]:
import os
import time
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
from matplotlib import pyplot as plt
from torchsummary import summary
import jammy_flows
from scipy.stats import norm

from helper import normalize, denormalize, train_model, get_normalized_data, evaluate_model, denormalize_std
from model_examples import TinyCNN

In [ ]:
DATA_PATH = "../data"

learning_rate = 0.8e-5
batch_size = 64
num_epochs = 100
patience = 7

train_fraction = 0.7
val_fraction = 0.15

seed = 42

torch.manual_seed(seed)
np.random.seed(seed)

In [ ]:
# Load normalized spectra and labels
spectra, labels, spectra_length, n_labels, labelNames, ranges = get_normalized_data(DATA_PATH)

print("spectra shape:", spectra.shape)
print("labels shape:", labels.shape)
print("spectra length:", spectra_length)
print("n_labels:", n_labels)
print("label names:", labelNames)

In [ ]:
# Convert to tensors
spectra_tensor = torch.tensor(spectra, dtype=torch.float32)
labels_tensor = torch.tensor(labels, dtype=torch.float32)

# Split into train / val / test
total_samples = len(spectra_tensor)
train_size = int(train_fraction * total_samples)
val_size = int(val_fraction * total_samples)
test_size = total_samples - train_size - val_size

dataset = TensorDataset(spectra_tensor, labels_tensor)

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(seed)
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"train: {len(train_dataset)}")
print(f"val:   {len(val_dataset)}")
print(f"test:  {len(test_dataset)}")

In [ ]:
# Device
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Using device:", device)

In [ ]:
# Encoder model
class TinyCNNEncoder(nn.Module):
    def __init__(self, latent_dimension):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.BatchNorm1d(16),
            nn.Dropout(0.1),
            nn.AvgPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.1),
            nn.AvgPool1d(2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.1),
            nn.AvgPool1d(2),

            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.1),
            nn.AvgPool1d(2),

            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.2),

            nn.AdaptiveAvgPool1d(32)
        )

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dimension)
        )

    def forward(self, x):
        if x.ndim == 2:
            x = x.unsqueeze(1)
        elif x.ndim != 3:
            raise ValueError(f"Unexpected input shape: {x.shape}")

        x = self.features(x)
        x = self.head(x)
        return x

In [ ]:
# Combined model
class CombinedModel(nn.Module):
    def __init__(self, encoder, nf_type="diagonal_gaussian", fp64_on_cpu=False):
        super().__init__()

        opt_dict = {}
        opt_dict["t"] = {}

        if nf_type == "diagonal_gaussian":
            opt_dict["t"]["cov_type"] = "diagonal"
            flow_defs = "t"
        elif nf_type == "full_gaussian":
            opt_dict["t"]["cov_type"] = "full"
            flow_defs = "t"
        elif nf_type == "full_flow":
            opt_dict["t"]["cov_type"] = "full"
            flow_defs = "gggt"
        else:
            raise Exception(f"Unknown nf type: {nf_type}")

        opt_dict["g"] = {}
        opt_dict["g"]["fit_normalization"] = 1
        opt_dict["g"]["upper_bound_for_widths"] = 1.0
        opt_dict["g"]["lower_bound_for_widths"] = 0.01

        self.nf_type = nf_type
        self.fp64_on_cpu = fp64_on_cpu

        self.pdf = jammy_flows.pdf(
            "e3",
            flow_defs,
            options_overwrite=opt_dict,
            amortize_everything=True,
            amortization_mlp_use_custom_mode=True
        )

        num_flow_parameters = self.pdf.total_number_amortizable_params
        print("The normalizing flow has", num_flow_parameters, "parameters")

        self.encoder = encoder(num_flow_parameters)

    def log_pdf_evaluation(self, target_labels, input_data):
        latent_intermediate = self.encoder(input_data)

        if self.nf_type == "full_flow":
            if self.fp64_on_cpu:
                latent_intermediate = latent_intermediate.cpu().to(torch.float64)
                target_labels = target_labels.cpu().to(torch.float64)
            else:
                latent_intermediate = latent_intermediate.to(torch.float64)
                target_labels = target_labels.to(torch.float64)

        log_pdf, _, _ = self.pdf(target_labels, amortization_parameters=latent_intermediate)
        return log_pdf

    def sample(self, flow_params, samplesize_per_batchitem=1000):
        if self.nf_type == "full_flow":
            if self.fp64_on_cpu:
                flow_params = flow_params.cpu().to(torch.float64)
            else:
                flow_params = flow_params.to(torch.float64)

        batch_size = flow_params.shape[0]

        repeated_samples, _, _, _ = self.pdf.sample(
            amortization_parameters=flow_params.repeat_interleave(samplesize_per_batchitem, dim=0),
            allow_gradients=False
        )

        reshaped_samples = repeated_samples[:, None, :].view(
            batch_size, samplesize_per_batchitem, -1
        )

        return reshaped_samples

    def forward(self, input_data, samplesize_per_batchitem=1000):
        flow_params = self.encoder(input_data)
        samples = self.sample(flow_params, samplesize_per_batchitem=samplesize_per_batchitem)

        means = samples.mean(dim=1)
        std_deviations = samples.std(dim=1)

        return torch.cat([means, std_deviations], dim=1)

    def visualize_pdf(self, input_data, filename, samplesize=1000, batch_index=0, truth=None):
        input_bitem = input_data[batch_index:batch_index+1]
        flow_params = self.encoder(input_bitem)
        samples = self.sample(flow_params, samplesize_per_batchitem=samplesize)
        samples = samples.squeeze(0)

        mean = samples.mean(dim=0).cpu().numpy()
        std = samples.std(dim=0).cpu().numpy()
        samples = samples.cpu().numpy()

        fig, axdict = plt.subplots(3, 1, figsize=(6, 10))

        for dim_ind in range(3):
            axdict[dim_ind].hist(
                samples[:, dim_ind],
                color="k",
                density=True,
                bins=50,
                alpha=0.5,
                label="density based on samples"
            )

            min_sample = samples[:, dim_ind].min()
            max_sample = samples[:, dim_ind].max()
            xvals = np.linspace(min_sample, max_sample, 1000)
            yvals = norm.pdf(xvals, loc=mean[dim_ind], scale=std[dim_ind])
            axdict[dim_ind].plot(xvals, yvals, color="green", label="Gaussian approximation")

            if truth is not None:
                true_value = truth[dim_ind].cpu().item()
                axdict[dim_ind].axvline(true_value, color="red", label="true value")

            axdict[dim_ind].set_title(labelNames[dim_ind])

            if dim_ind == 0:
                axdict[dim_ind].legend()

        plt.tight_layout()
        plt.savefig(filename)
        plt.show()

In [ ]:
# Normalizing flow NLL loss
def nf_nll_loss(inputs, batch_labels, model):
    log_pdfs = model.log_pdf_evaluation(batch_labels, inputs)
    return -log_pdfs.mean()

def loss_function(inputs, labels, model):
    loss_result = nf_nll_loss(inputs, labels, model)
    return loss_result

In [ ]:
# Train and evaluate the model, based on flow type
def run_flow_experiment(normalizing_flow_type):
    print("\n" + "="*80)
    print(f"Running flow type: {normalizing_flow_type}")
    print("="*80)

    fp64_on_cpu = False
    if normalizing_flow_type == "full_flow" and device.type == "mps":
        fp64_on_cpu = True

    print("Using device:", device)
    print("Performing fp64 on CPU:", fp64_on_cpu)

    model = CombinedModel(
        TinyCNNEncoder,
        nf_type=normalizing_flow_type,
        fp64_on_cpu=fp64_on_cpu
    )

    try:
        summary(model.encoder, input_size=(spectra_length,))
    except Exception as e:
        print("Could not print model summary:", e)

    model = model.to(device)

    xb, yb = next(iter(train_loader))
    xb = xb.to(device)
    yb = yb.to(device)

    model.eval()
    with torch.no_grad():
        log_prob = model.log_pdf_evaluation(yb, xb)
        loss = -log_prob.mean()

    print("Sanity check log_prob shape:", log_prob.shape)
    print("Sanity check loss:", loss.item())

    train_losses, val_losses, best_model = train_model(
        model,
        train_loader,
        val_loader,
        loss_function,
        learning_rate,
        num_epochs,
        patience,
        device,
        model_name=f"tiny_CNN_normalizing_flow_{normalizing_flow_type}"
    )

    model.load_state_dict(best_model)
    model.to(device)
    model.eval()

    all_predictions, all_true_labels, _, _ = evaluate_model(model, test_loader, loss_function, device)

    all_predictions = torch.tensor(all_predictions, dtype=torch.float32)
    all_true_labels = torch.tensor(all_true_labels, dtype=torch.float32)

    all_mean = all_predictions[:, :n_labels]
    all_std = all_predictions[:, n_labels:]

    all_mean_denorm = denormalize(all_mean, ranges)
    all_true_denorm = denormalize(all_true_labels, ranges)
    all_std_denorm = denormalize_std(all_std, ranges)

    pulls = (all_mean_denorm - all_true_denorm) / (all_std_denorm + 1e-8)

    os.makedirs("plots", exist_ok=True)

    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_losses) + 1), train_losses, label="Training loss")
    plt.plot(range(1, len(val_losses) + 1), val_losses, label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Normalizing flow NLL loss")
    plt.title(f"Training and validation loss: {normalizing_flow_type}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"plots/{normalizing_flow_type}_training_validation_loss.png")
    plt.show()

    plt.figure(figsize=(16, 7.5))
    for j in range(n_labels):
        plt.subplot(1, n_labels, j + 1)
        gt = all_true_denorm[:, j].numpy()
        pred = all_mean_denorm[:, j].numpy()
        plt.scatter(gt, pred, s=6, alpha=0.2)
        mn = min(gt.min(), pred.min())
        mx = max(gt.max(), pred.max())
        plt.plot([mn, mx], [mn, mx], "k--", label="Perfect prediction")
        plt.xlabel(f"true {labelNames[j]}")
        plt.ylabel(f"predicted {labelNames[j]}")
        plt.legend()
    plt.tight_layout()
    plt.savefig(f"plots/{normalizing_flow_type}_scatter.png")
    plt.show()

    plt.figure(figsize=(16, 4))
    for j in range(n_labels):
        plt.subplot(1, n_labels, j + 1)
        p = pulls[:, j].numpy()

        mean_pull = p.mean()
        std_pull = p.std()

        plt.hist(p, bins=40, density=True, alpha=0.8)

        x = np.linspace(-5, 5, 400)
        y = (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * x**2)
        plt.plot(x, y, "r-", label="standard normal")

        plt.title(f"{labelNames[j]}\nmean={mean_pull:.3f}, std={std_pull:.3f}")
        plt.xlabel("pull")
        plt.ylabel("density")
        plt.legend()

    plt.tight_layout()
    plt.savefig(f"plots/{normalizing_flow_type}_pull_distributions.png")
    plt.show()

    for j, name in enumerate(labelNames):
        p = pulls[:, j].numpy()
        print(f"{name:>8s} | pull mean: {p.mean():.4f} | pull std: {p.std():.4f}")

    xb, yb = next(iter(test_loader))
    xb = xb.to(device)

    model.visualize_pdf(
        xb,
        filename=f"plots/{normalizing_flow_type}_pdf_example_0.png",
        samplesize=5000,
        batch_index=0,
        truth=yb[0]
    )

    model.visualize_pdf(
        xb,
        filename=f"plots/{normalizing_flow_type}_pdf_example_1.png",
        samplesize=5000,
        batch_index=1,
        truth=yb[1]
    )

    return {
        "flow_type": normalizing_flow_type,
        "train_losses": train_losses,
        "val_losses": val_losses,
        "all_mean": all_mean,
        "all_std": all_std,
        "all_true_labels": all_true_labels,
        "all_mean_denorm": all_mean_denorm,
        "all_std_denorm": all_std_denorm,
        "all_true_denorm": all_true_denorm,
        "pulls": pulls,
    }

In [ ]:
flow_types = ["diagonal_gaussian", "full_gaussian", "full_flow"]

results = {}
for flow_type in flow_types:
    results[flow_type] = run_flow_experiment(flow_type)

In [ ]:
for flow_type in flow_types:
    pulls = results[flow_type]["pulls"]
    print("\n" + "-"*60)
    print(flow_type)
    print("-"*60)
    for j, name in enumerate(labelNames):
        p = pulls[:, j].numpy()
        print(f"{name:>8s} | pull mean: {p.mean():.4f} | pull std: {p.std():.4f}")